# 36 · Transfer methods from the literature scan (E89-E92)

Leave one platform out on TikTok / YouTube / X; baseline ridge on log likes + log comments. Sources: `docs/en/literature_notes.md` (methods section).

In [1]:
import sys, json, warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw" / "reach_samples_students.csv").exists())
sys.path.insert(0, str(ROOT / "src"))
pd.set_option("display.width", 250, "display.max_columns", 30, "display.max_rows", 400)
import extra_models as em, history as H, transfer_lit as tl

In [2]:
f, meta = em.load(ROOT)

## E89 · Few-shot recalibration with k target posts
k posts drawn by brand (at most 2 per brand), 200 draws; scored on target posts of other brands.

In [3]:
F89 = tl.fewshot(f, meta); display(F89[F89.MAE.notna()].pivot_table(index="method", columns=["platform", "k"], values="MAE").round(3))
F89[F89.coverage.notna()].pivot_table(index="method", columns=["platform", "k"], values=["coverage", "half-width (×)"]).round(2)

platform                                          tiktok                                    twitter                                    youtube                                   
k                                                    0      5      10     25     50     100     0      5      10     25     50     100     0      5      10     25     50     100
method                                                                                                                                                                           
biased ridge to source weights                       NaN  0.348  0.326  0.313  0.307  0.304     NaN  0.604  0.579  0.550  0.532  0.525     NaN  0.681  0.631  0.616  0.606  0.602
intercept + slope (slope shrunk to 1)                NaN  0.359  0.334  0.317  0.309  0.306     NaN  0.598  0.567  0.542  0.533  0.528     NaN  0.713  0.669  0.629  0.619  0.610
mean shift                                           NaN  0.341  0.331  0.320  0.315  0.314     NaN  0.603  0.574  0.556  0.551  0.547     NaN  0.688  0.664  0.631  0.626  0.620
median shift                                         NaN  0.347  0.332  0.321  0.315  0.314     NaN  0.621  0.585  0.560  0.555  0.550     NaN  0.679  0.641  0.618  0.609  0.607
shrunk mean shift (EB)                               NaN  0.316  0.317  0.316  0.314  0.313     NaN  0.637  0.632  0.620  0.604  0.584     NaN  0.640  0.640  0.625  0.624  0.619
shrunk median shift (EB)                             NaN  0.317  0.318  0.317  0.314  0.314     NaN  0.637  0.632  0.619  0.602  0.581     NaN  0.636  0.626  0.615  0.609  0.607
shrunk median shift (EB, tau2 from all six pairs)    NaN  0.315  0.316  0.316  0.314  0.313     NaN  0.588  0.574  0.558  0.553  0.549     NaN  0.626  0.621  0.614  0.608  0.607
zero-shot ridge                                    0.312    NaN    NaN    NaN    NaN    NaN   0.642    NaN    NaN    NaN    NaN    NaN   0.604    NaN    NaN    NaN    NaN    NaN

coverage                                                                                         half-width (×)                                                                                               \
platform                                   tiktok                         twitter                         youtube                                 tiktok                             twitter                             youtube                        
k                                             5     10    25    50    100     5     10    25    50    100     5     10    25    50    100            5      10     25     50     100     5      10     25     50     100     5      10     25     50    
method                                                                                                                                                                                                                                                  
90% interval: few-shot conformal             0.77  0.89  0.92  0.89  0.89    0.77  0.87  0.92  0.90  0.90    0.76  0.90  0.91  0.90  0.89           3.50   5.41   5.70   4.48   4.44   10.01  16.33  18.37  15.13  14.25   11.90  48.52  49.24  29.99   
90% interval: source conformal, no shift     0.99  0.99  0.99  0.99  0.99    0.81  0.81  0.81  0.81  0.81    0.79  0.79  0.79  0.79  0.79          18.06  18.06  18.06  18.06  18.06   10.56  10.56  10.56  10.56  10.56    7.82   7.82   7.82   7.82   

                                                 
platform                                         
k                                           100  
method                                           
90% interval: few-shot conformal          28.16  
90% interval: source conformal, no shift   7.82

## E90 · Anchor regression, group DRO, stabilised weights (no target labels)

In [4]:
P90 = tl.anchor_dro(f, meta); S90 = H.summarise(P90); display(S90.pivot(index="model", columns="platform", values="MAE")[tl.PLATS].round(3)); display(S90.pivot(index="model", columns="platform", values="bias")[tl.PLATS].round(3))
G90 = pd.DataFrame([{"platform": p, "model": k, **dict(zip(["gain", "CI low", "CI high"], H.gain_ci(P90, p, "anchor gamma=1", k)))} for p in tl.PLATS for k in P90.model.unique() if k != "anchor gamma=1"]); G90.round(3)

platform,tiktok,youtube,twitter
model,,,
anchor gamma=0,2.266,2.610,2.449
anchor gamma=0.5,0.307,0.602,0.640
anchor gamma=1,0.312,0.604,0.641
anchor gamma=10,0.390,0.620,0.649
anchor gamma=100,0.440,0.625,0.654
anchor gamma=2,0.325,0.609,0.643
anchor gamma=5,0.360,0.616,0.646
group DRO (squared loss),0.302,0.721,0.654
weights: classifier (E88),0.385,0.602,0.618


platform,tiktok,youtube,twitter
model,,,
anchor gamma=0,-2.266,-2.610,-2.449
anchor gamma=0.5,-0.079,-0.207,0.381
anchor gamma=1,-0.052,-0.206,0.382
anchor gamma=10,0.093,-0.203,0.394
anchor gamma=100,0.157,-0.202,0.402
anchor gamma=2,-0.013,-0.205,0.385
anchor gamma=5,0.050,-0.203,0.390
group DRO (squared loss),-0.009,-0.044,0.399
weights: classifier (E88),0.083,-0.092,0.335


,platform,model,gain,CI low,CI high
0,tiktok,anchor gamma=0,-1.954,-1.974,-1.936
1,tiktok,anchor gamma=0.5,0.005,0.003,0.007
2,tiktok,anchor gamma=2,-0.013,-0.016,-0.011
3,tiktok,anchor gamma=5,-0.048,-0.054,-0.042
4,tiktok,anchor gamma=10,-0.077,-0.086,-0.069
5,tiktok,anchor gamma=100,-0.128,-0.141,-0.116
6,tiktok,group DRO (squared loss),0.010,0.005,0.016
7,tiktok,weights: classifier (E88),-0.073,-0.082,-0.064
8,tiktok,weights: flattened ^0.5,-0.020,-0.025,-0.015
9,tiktok,weights: flattened ^0.25,-0.004,-0.006,-0.002


## E91 · Input subset chosen by source-to-source transfer

In [5]:
T91 = tl.subset_search(f, meta)
sel = T91.loc[T91.groupby("platform")["source-to-source MAE"].idxmin()].assign(pick="chosen by sources")
ora = T91.loc[T91.groupby("platform")["target MAE"].idxmin()].assign(pick="best on target (oracle)")
ref = T91[T91.subset == "log_likes+log_comments"].assign(pick="likes + comments")
all_ = T91[T91.subset == "+".join(tl.FEATS)].assign(pick="all six")
C91 = pd.concat([ref, sel, ora, all_]); display(C91.round(3))
print("Spearman source-to-source vs target MAE over subsets:", T91.groupby("platform").apply(lambda q: q["source-to-source MAE"].corr(q["target MAE"], method="spearman")).round(2).to_dict())

,platform,subset,n features,source-to-source MAE,target MAE,target bias,pick
6,tiktok,log_likes+log_comments,2,0.696,0.312,0.053,likes + comments
69,youtube,log_likes+log_comments,2,0.570,0.604,0.206,likes + comments
132,twitter,log_likes+log_comments,2,0.452,0.642,-0.385,likes + comments
51,tiktok,log_comments+log_shares+log_followers+cpl,4,0.585,0.497,-0.078,chosen by sources
132,twitter,log_likes+log_comments,2,0.452,0.642,-0.385,chosen by sources
99,youtube,log_comments+cpl+lpf,3,0.511,0.596,0.049,chosen by sources
13,tiktok,log_comments+cpl,2,0.673,0.305,0.052,best on target (oracle)
148,twitter,log_likes+log_comments+log_followers,3,0.558,0.583,-0.314,best on target (oracle)
88,youtube,log_likes+log_shares+log_followers,3,0.562,0.577,0.108,best on target (oracle)
62,tiktok,log_likes+log_comments+log_shares+log_follower...,6,0.687,0.476,-0.006,all six


Spearman source-to-source vs target MAE over subsets: {'tiktok': 0.24, 'twitter': 0.65, 'youtube': 0.58}


## E92 · Brand effect carried over from the source platforms

In [6]:
P92 = tl.brand_carry(f, meta); S92 = H.summarise(P92); display(S92.round(3))
G92 = pd.DataFrame([{"platform": p, "model": k, **dict(zip(["gain", "CI low", "CI high"], H.gain_ci(P92, p, "ridge, brands kept", k)))} for p in tl.PLATS for k in P92.model.unique() if k != "ridge, brands kept"]); G92.round(3)

,platform,model,posts,MAE,CI low,CI high,bias
0,tiktok,"ridge, brands kept",2491,0.312,0.300,0.324,-0.053
1,tiktok,"+ brand effect from sources, shrink k=1",2491,0.377,0.357,0.397,-0.052
2,tiktok,"+ brand effect from sources, shrink k=3",2491,0.360,0.343,0.377,-0.048
3,tiktok,"+ brand effect from sources, shrink k=10",2491,0.335,0.321,0.350,-0.045
4,youtube,"ridge, brands kept",2441,0.604,0.573,0.637,-0.206
5,youtube,"+ brand effect from sources, shrink k=1",2441,0.612,0.581,0.646,-0.206
6,youtube,"+ brand effect from sources, shrink k=3",2441,0.607,0.577,0.641,-0.205
7,youtube,"+ brand effect from sources, shrink k=10",2441,0.604,0.572,0.637,-0.205
8,twitter,"ridge, brands kept",1803,0.642,0.616,0.664,0.385
9,twitter,"+ brand effect from sources, shrink k=1",1803,0.664,0.632,0.693,0.381


,platform,model,gain,CI low,CI high
0,tiktok,"+ brand effect from sources, shrink k=1",-0.065,-0.083,-0.048
1,tiktok,"+ brand effect from sources, shrink k=3",-0.048,-0.062,-0.034
2,tiktok,"+ brand effect from sources, shrink k=10",-0.023,-0.032,-0.014
3,youtube,"+ brand effect from sources, shrink k=1",-0.008,-0.016,0.000
4,youtube,"+ brand effect from sources, shrink k=3",-0.003,-0.010,0.003
5,youtube,"+ brand effect from sources, shrink k=10",0.001,-0.004,0.006
6,twitter,"+ brand effect from sources, shrink k=1",-0.022,-0.044,-0.002
7,twitter,"+ brand effect from sources, shrink k=3",-0.015,-0.033,0.001
8,twitter,"+ brand effect from sources, shrink k=10",-0.007,-0.017,0.004


In [7]:
json.dump({"fewshot": F89.to_dict("records"), "anchor_dro_models": S90.to_dict("records"), "anchor_dro_gains": G90.to_dict("records"), "subsets": T91.to_dict("records"), "subset_choice": C91.to_dict("records"),
           "brand_carry_models": S92.to_dict("records"), "brand_carry_gains": G92.to_dict("records")}, open(ROOT / "data" / "processed" / "transfer_lit_numbers.json", "w"), indent=1, default=float)